# RAGAS Evaluation for QBrain

This notebook adds a secondary evaluation layer using RAGAS on top of the existing benchmark outputs.

It reads `benchmark_full_results.csv`, computes RAGAS metrics, and saves paper-ready tables.


In [ ]:
from pathlib import Path
import os
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
elif ROOT.name != "rag_lab":
    ROOT = Path("d:/Qbrainpython/QBrain/rag_lab")

BENCHMARK_CSV = ROOT / "data" / "outputs" / "evaluation" / "rag_benchmark" / "benchmark_full_results.csv"
OUT_TABLES = ROOT / "results" / "tables"
OUT_TABLES.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("BENCHMARK_CSV:", BENCHMARK_CSV)
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))


## 1) Install dependencies (run once)

If `ragas` is not installed, uncomment and run the next line.


In [ ]:
# %pip install ragas datasets langchain-openai -U


## 2) Prepare dataset from benchmark output


In [ ]:
from datasets import Dataset

df = pd.read_csv(BENCHMARK_CSV)

required_cols = ["question", "generated_answer", "expected_answer"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns in benchmark output: {missing}")

# Build contexts from retrieved file names (proxy context when raw chunks are not persisted in CSV).
def _contexts_from_row(row):
    # Preferred: real retrieved chunk texts saved by run_rag_benchmark.py
    raw_ctx = row.get("retrieved_contexts_json", None)
    if isinstance(raw_ctx, str) and raw_ctx.strip() and raw_ctx.strip() != "nan":
        import json
        try:
            arr = json.loads(raw_ctx)
            if isinstance(arr, list):
                return [str(x) for x in arr if str(x).strip()]
        except Exception:
            pass

    # Fallback (older CSVs): filenames only; this is weaker for faithfulness
    val = str(row.get("retrieved_files_topk", ""))
    if not val or val == "nan":
        return []
    return [x.strip() for x in val.split(";") if x.strip()]

ragas_df = pd.DataFrame(
    {
        "question": df["question"].astype(str),
        "answer": df["generated_answer"].astype(str),
        "ground_truth": df["expected_answer"].astype(str),
        "contexts": df.apply(_contexts_from_row, axis=1),
        "srs_file": df["srs_file"].astype(str),
        "question_id": df["question_id"].astype(str),
    }
)

ragas_dataset = Dataset.from_pandas(ragas_df[["question", "answer", "ground_truth", "contexts"]], preserve_index=False)
display(ragas_df.head(3))
print("Rows:", len(ragas_df))


## 3) Run RAGAS metrics

Default metrics below are commonly reported in papers:
- `faithfulness`
- `answer_relevancy`
- `answer_correctness`


In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, answer_correctness
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
emb = OpenAIEmbeddings(model="text-embedding-3-small")

result = evaluate(
    ragas_dataset,
    metrics=[faithfulness, answer_relevancy, answer_correctness],
    llm=llm,
    embeddings=emb,
)

result_df = result.to_pandas()
display(result_df.head())
print("Columns:", list(result_df.columns))


## 4) Export paper-ready tables


In [ ]:
full = pd.concat([ragas_df.reset_index(drop=True), result_df.reset_index(drop=True)], axis=1)

metric_cols = [c for c in ["faithfulness", "answer_relevancy", "answer_correctness"] if c in full.columns]
if not metric_cols:
    raise ValueError("RAGAS metric columns not found in result dataframe.")

overall = full[metric_cols].mean().reset_index()
overall.columns = ["metric", "value"]

per_srs = full.groupby("srs_file", as_index=False)[metric_cols].mean()

full.to_csv(OUT_TABLES / "ragas_by_question.csv", index=False)
overall.to_csv(OUT_TABLES / "ragas_overall_summary.csv", index=False)
per_srs.to_csv(OUT_TABLES / "ragas_summary_by_srs.csv", index=False)

(OUT_TABLES / "ragas_overall_summary.tex").write_text(
    overall.to_latex(index=False, float_format=lambda x: f"{x:.4f}"),
    encoding="utf-8",
)
(OUT_TABLES / "ragas_summary_by_srs.tex").write_text(
    per_srs.to_latex(index=False, float_format=lambda x: f"{x:.4f}"),
    encoding="utf-8",
)

display(overall)
display(per_srs)
print("Saved RAGAS outputs to:", OUT_TABLES)


## 5) Suggested reporting text (for paper)

Use this as a template in Results:

- "We complemented similarity-threshold evaluation with RAGAS metrics (faithfulness, answer relevancy, and answer correctness)."
- "This provides an additional quality perspective beyond strict semantic-threshold accuracy and helps distinguish factual grounding from stylistic variance."
